# RQ06 — Correção de Holm e tamanho de efeito (#172)

Este notebook completa a análise por fatores da #171: corrige comparações múltiplas e reporta **ε² para Kruskal-Wallis** e **Cliff's delta para Mann-Whitney**. A unidade continua sendo o repositório, com quartis fixos de estrelas, contribuidores e idade em dias, nas seis variantes de métricas.

A entrada é somente o CSV final de 500 repositórios. Os testes brutos, grupos e cortes são recalculados pelas funções da #171; seus arquivos históricos permanecem preservados. A interpretação dos resultados para o artigo pertence à #177.

## Reprodução

Instale `requirements.txt` e execute todas as células a partir de `Lab03/` ou `Lab03/analise/`. Para outra exportação do mesmo contrato, defina `LAB03_RQ06_HOLM_CSV`; para uma pasta isolada de saída, defina `LAB03_RQ06_HOLM_SAIDA`. Caminhos relativos usam a pasta de trabalho do kernel. Nenhum comando consulta a API ou exige token.

Os quartis e as exclusões seguem a #171: cortes calculados antes das ausências por métrica, empates não separados por ID, valores zero preservados, dados inválidos recusados e mínimo de cinco observações válidas em cada grupo para testes assintóticos. Os valores das métricas não são imputados para produzir um resultado.

In [1]:
import hashlib
import json
import os
import platform
import sys
from pathlib import Path

import numpy as np
import pandas as pd
import scipy
import statsmodels
from IPython.display import Markdown, display

candidatas = [p / sufixo for p in (Path.cwd(), *Path.cwd().parents) for sufixo in ("", "Lab03")]
RAIZ = next(p for p in candidatas if (p / "pipeline/inferencia_rq06.py").is_file())
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from pipeline.comparacao_rq06 import ALPHA, FATORES, METRICAS, carregar_dataset
from pipeline.inferencia_rq06 import analisar

DATASET = Path(os.environ.get("LAB03_RQ06_HOLM_CSV", str(RAIZ / "entregas/s02-03-dataset-500-final/dataset-500.csv"))).expanduser().resolve()
SAIDA = Path(os.environ.get("LAB03_RQ06_HOLM_SAIDA", str(RAIZ / "analise"))).expanduser().resolve()
TABELAS = SAIDA / "tabelas"
ARQUIVOS = {chave: f"rq06_holm_{chave}.csv" for chave in ("globais", "pares", "familias")}
destinos = {TABELAS / nome for nome in (*ARQUIVOS.values(), "rq06_holm_proveniencia.json")}
if DATASET in {p.resolve() for p in destinos}:
    raise ValueError("Saída não pode sobrescrever o CSV de entrada.")
TABELAS.mkdir(parents=True, exist_ok=True)
dados = carregar_dataset(DATASET)
sha_entrada = dados.attrs["csv_sha256"]
pd.options.display.float_format = "{:.6g}".format
resultados = analisar(dados, alpha=ALPHA)
globais, pares = resultados["globais"], resultados["pares"]
print(f"Repositórios: {len(dados)} | α por família: {ALPHA} | SHA-256: {sha_entrada}")
display(resultados["base"]["cortes"])
display(resultados["familias"])

Repositórios: 500 | α por família: 0.05 | SHA-256: 2552ba6b0779e895d01bea88742cb668ee1812a8369a224ed5281708140b30bd


,fator,n_populacao,n_fator,ausentes_fator,grupo,limite_inferior,limite_superior,inclui_inferior,n_grupo,grupos_efetivos,motivo
0,estrelas,500,500,0,Q1,30088,36047.8,True,125,4,None
1,estrelas,500,500,0,Q2,36047.8,45049,False,125,4,None
2,estrelas,500,500,0,Q3,45049,68388.5,False,125,4,None
3,estrelas,500,500,0,Q4,68388.5,391470,False,125,4,None
4,contribuidores,500,500,0,Q1,1,180.75,True,125,4,None
5,contribuidores,500,500,0,Q2,180.75,395,False,125,4,None
6,contribuidores,500,500,0,Q3,395,887.25,False,125,4,None
7,contribuidores,500,500,0,Q4,887.25,9944,False,125,4,None
8,idade_dias,500,500,0,Q1,12,939,True,125,4,None
9,idade_dias,500,500,0,Q2,939,2390.5,False,125,4,None


,familia,testes_planejados,testes_estimaveis,testes_nao_estimaveis,rejeicoes_holm,alpha,metodo
0,globais,18,18,0,7,0.05,holm
1,pares,108,108,0,27,0.05,holm


## Famílias e decisões de Holm

Usamos `statsmodels.stats.multitest.multipletests(..., method="holm")`, com α=0,05 e resultados mantidos na ordem original. O procedimento ajusta p-valores por família; estes permanecem registrados ao lado dos valores brutos.

1. **Família global:** todas as combinações de três fatores × seis métricas, **18 hipóteses** neste CSV. O teste é Kruskal-Wallis com três ou mais grupos; se os quartis colapsarem em dois grupos, usa-se Mann-Whitney direto, na mesma família global.
2. **Família dos pares:** todos os pares não ordenados de grupos dos fatores com três ou mais grupos, em todas as métricas, **108 hipóteses** neste CSV (3×6×6). Os p-valores são calculados e corrigidos **antes de selecionar o relato**, inclusive para globais que não foram rejeitados. Isso amplia a correção em relação aos 48 pares selecionados nominalmente na #171. Não se escolhe a família pelos resultados observados.

`rejeita_holm` segue a decisão do statsmodels, equivalente ao p ajustado ≤ α. Um par tem `relatar_diferenca=True` somente se **o global e o par** forem rejeitados nas respectivas famílias. Os demais pares continuam disponíveis para auditoria. O controle é declarado por família, não como uma única família de 126 hipóteses; não inclui os testes da RQ05.

Hipóteses não estimáveis permanecem na família planejada. Usam p=1 **apenas como marcador interno conservador para a correção**, restaurando p corrigido e decisão como ausentes na saída. Esse marcador não é um p-valor observado. Em outro CSV, o número de grupos e de contrastes planejados é registrado conforme os fatores disponíveis, antes de qualquer seleção por significância.

Referência: [multipletests no statsmodels](https://www.statsmodels.org/stable/generated/statsmodels.stats.multitest.multipletests.html).

In [2]:
display(globais[["fator", "metrica", "teste", "n", "estatistica", "p_valor_bruto", "p_valor_holm", "rejeita_holm", "poshoc_elegivel_holm", "motivo"]])
print(f"Globais rejeitados após Holm: {int(globais['rejeita_holm'].sum())}/{len(globais)}")
# A coluna histórica poshoc_executado continua representando a seleção nominal da #171.
# poshoc_elegivel_holm é a decisão global após correção nesta #172.
mudaram = globais.loc[globais["diferenca_nominal"].fillna(False) & ~globais["rejeita_holm"].fillna(False)]
display(Markdown("### Diferenças nominais que não se mantêm após correção"))
display(mudaram[["fator", "metrica", "p_valor_bruto", "p_valor_holm"]])

,fator,metrica,teste,n,estatistica,p_valor_bruto,p_valor_holm,rejeita_holm,poshoc_elegivel_holm,motivo
0,estrelas,deployment_frequency,kruskal_wallis,500,8.13425,0.0433171,0.476489,False,False,None
1,estrelas,lead_time_a,kruskal_wallis,499,1.71584,0.633417,1,False,False,None
2,estrelas,lead_time_b,kruskal_wallis,499,1.16333,0.761812,1,False,False,None
3,estrelas,cfr_a,kruskal_wallis,500,3.69743,0.296044,1,False,False,None
4,estrelas,cfr_b,kruskal_wallis,498,5.13504,0.162172,1,False,False,None
5,estrelas,tempo_recuperacao,kruskal_wallis,469,4.32817,0.228139,1,False,False,None
6,contribuidores,deployment_frequency,kruskal_wallis,500,3.1869,0.363697,1,False,False,None
7,contribuidores,lead_time_a,kruskal_wallis,499,74.2056,5.36232e-16,7.50725e-15,True,True,None
8,contribuidores,lead_time_b,kruskal_wallis,499,108.712,2.07715e-23,3.32344e-22,True,True,None
9,contribuidores,cfr_a,kruskal_wallis,500,4.47961,0.214116,1,False,False,None


Globais rejeitados após Holm: 7/18


### Diferenças nominais que não se mantêm após correção

,fator,metrica,p_valor_bruto,p_valor_holm
0,estrelas,deployment_frequency,0.0433171,0.476489


## Efeito global: ε²

Seguimos a fórmula explícita do enunciado: **ε² = H / (n − 1)**, com H corrigido para empates e **n efetivamente usado naquele fator × métrica**, após as exclusões. A medida é não negativa e limitada a [0,1]; é reportada em todos os Kruskal-Wallis estimáveis, mesmo quando o p ajustado não rejeita o teste.

Não se substitui essa convenção por `(H − k + 1)/(n − k)`, que produziria outro estimador. Não se atribuem cortes qualitativos de magnitude a ε², pois o enunciado só os especifica para o delta. Não estimável ou não aplicável (Mann-Whitney direto) permanece ausente. Correções de no máximo 1e−12 nos limites de H são toleradas apenas para ruído numérico; valores materialmente fora do domínio são recusados.

ε² não mede causalidade nem a porcentagem de variação explicada na escala original das métricas.

In [3]:
display(globais[["fator", "metrica", "n", "estatistica", "epsilon2", "p_valor_holm", "rejeita_holm"]])

,fator,metrica,n,estatistica,epsilon2,p_valor_holm,rejeita_holm
0,estrelas,deployment_frequency,500,8.13425,0.0163011,0.476489,False
1,estrelas,lead_time_a,499,1.71584,0.00344547,1,False
2,estrelas,lead_time_b,499,1.16333,0.002336,1,False
3,estrelas,cfr_a,500,3.69743,0.00740969,1,False
4,estrelas,cfr_b,498,5.13504,0.0103321,1,False
5,estrelas,tempo_recuperacao,469,4.32817,0.00924823,1,False
6,contribuidores,deployment_frequency,500,3.1869,0.00638656,1,False
7,contribuidores,lead_time_a,499,74.2056,0.149007,7.50725e-15,True
8,contribuidores,lead_time_b,499,108.712,0.218298,3.32344e-22,True
9,contribuidores,cfr_a,500,4.47961,0.00897717,1,False


## Efeito dos pares: Cliff's delta

Calculamos **δ = 2·U₁/(n₁·n₂) − 1**, com o U associado ao **primeiro grupo**, na ordem crescente dos quartis. δ>0 significa valores numéricos maiores no grupo 1; δ<0, maiores no grupo 2. Empates contribuem com metade para U e zero para a diferença entre proporções de vitórias e derrotas. Inverter as amostras inverte o sinal do delta e preserva seu módulo.

O sinal não significa automaticamente “melhor”: DF maior e tempo/CFR menor têm sentidos distintos. A magnitude usa |δ|, conforme os cortes do enunciado: **<0,147 desprezível; <0,33 pequeno; <0,474 médio; ≥0,474 grande**. Efeito zero é preservado como zero; efeito não estimável continua ausente. Com apenas dois grupos válidos, o delta aparece na tabela global, com identificação e tamanho dos dois grupos, sem duplicar a hipótese na família de pós-testes.

A correção de Holm modifica os p-valores, não U, H ou os tamanhos de efeito. Os deltas são disponibilizados para todos os pares estimáveis, não só os selecionados para relato.

Referência para a orientação do U: [Mann-Whitney no SciPy](https://docs.scipy.org/doc/scipy/reference/generated/scipy.stats.mannwhitneyu.html).

In [4]:
selecionados = pares.loc[pares["relatar_diferenca"]].copy()
colunas = ["fator", "metrica", "grupo_1", "grupo_2", "n_1", "n_2", "U_1", "cliff_delta", "magnitude_delta", "p_valor_bruto", "p_valor_holm"]
display(selecionados[colunas])
print(f"Pares com rejeição própria após Holm: {int(pares['rejeita_holm'].sum())}/{len(pares)}")
print(f"Pares que também passam pelo global corrigido: {len(selecionados)}")
display(selecionados.groupby(["fator", "metrica"], sort=False).size().rename("pares_relataveis").to_frame())

,fator,metrica,grupo_1,grupo_2,n_1,n_2,U_1,cliff_delta,magnitude_delta,p_valor_bruto,p_valor_holm
43,contribuidores,lead_time_a,Q1,Q3,125,125,5195,-0.33504,medio,4.70113e-06,0.0004184
44,contribuidores,lead_time_a,Q1,Q4,125,124,3254,-0.580129,grande,2.55278e-15,2.60383e-13
46,contribuidores,lead_time_a,Q2,Q4,125,124,4410,-0.430968,medio,4.18433e-09,4.01696e-07
47,contribuidores,lead_time_a,Q3,Q4,125,124,5206,-0.328258,pequeno,7.60676e-06,0.000669395
49,contribuidores,lead_time_b,Q1,Q3,125,125,4647,-0.405184,medio,3.08977e-08,2.87348e-06
50,contribuidores,lead_time_b,Q1,Q4,125,124,2433,-0.686065,grande,8.29752e-21,8.79537e-19
52,contribuidores,lead_time_b,Q2,Q4,125,124,3475,-0.551613,grande,5.38963e-14,5.38963e-12
53,contribuidores,lead_time_b,Q3,Q4,125,124,4785,-0.382581,medio,1.82017e-07,1.63815e-05
73,idade_dias,deployment_frequency,Q1,Q3,125,125,10867.5,0.39104,medio,9.08558e-08,8.35873e-06
74,idade_dias,deployment_frequency,Q1,Q4,125,125,11011.5,0.409472,medio,2.18567e-08,2.05453e-06


Pares com rejeição própria após Holm: 27/108
Pares que também passam pelo global corrigido: 27


pares_relataveis
fator          metrica                               
contribuidores lead_time_a                          4
               lead_time_b                          4
idade_dias     deployment_frequency                 3
               lead_time_a                          6
               lead_time_b                          5
               cfr_b                                3
               tempo_recuperacao                    2

## Conferência da transição da #171 para a #172

Os testes globais, cortes, n e p-valores brutos da etapa anterior são preservados. Todos os 48 pares nominais da execução anterior estão presentes na família ampliada de 108; o indicador `executado_na_171` identifica esses pares. A tabela completa exportada inclui os novos contrastes necessários à correção da família, sem declarar diferenças quando o global corrigido não foi rejeitado.

In [5]:
originais = resultados["base"]["pares"].set_index(["fator", "metrica", "grupo_1", "grupo_2"])
novos = pares.set_index(originais.index.names)
np.testing.assert_allclose(novos.loc[originais.index, "p_valor_bruto"], originais["p_valor_bruto"])
np.testing.assert_allclose(novos.loc[originais.index, "U_1"], originais["U_1"])
print(f"Conferidos {len(originais)} pares nominais, sem mudança de U ou p bruto.")
# Resultados desta execução, calculados sem números fixos na célula.
linhas = [f"- {r.fator} × {r.metrica}: H={r.estatistica:.6g}; ε²={r.epsilon2:.6g}; p Holm={r.p_valor_holm:.6g}; n={r.n}."
          for r in globais.loc[globais["teste"].eq("kruskal_wallis") & globais["rejeita_holm"].fillna(False)].itertuples(index=False)]
display(Markdown("\n".join(linhas) if linhas else "Nenhum Kruskal-Wallis rejeitado após Holm nesta execução."))

Conferidos 48 pares nominais, sem mudança de U ou p bruto.


- contribuidores × lead_time_a: H=74.2056; ε²=0.149007; p Holm=7.50725e-15; n=499.
- contribuidores × lead_time_b: H=108.712; ε²=0.218298; p Holm=3.32344e-22; n=499.
- idade_dias × deployment_frequency: H=44.1678; ε²=0.0885127; p Holm=1.80731e-08; n=500.
- idade_dias × lead_time_a: H=140.775; ε²=0.28268; p Holm=4.37355e-29; n=499.
- idade_dias × lead_time_b: H=147.189; ε²=0.295561; p Holm=1.9157e-30; n=499.
- idade_dias × cfr_b: H=84.9685; ε²=0.170963; p Holm=3.95366e-17; n=498.
- idade_dias × tempo_recuperacao: H=18.2762; ε²=0.0390518; p Holm=0.00462909; n=469.

## Limites de interpretação

Holm reduz rejeições por múltiplas comparações, mas não elimina viés da amostra, dependência entre projetos ou confusão entre fatores. Ausência de rejeição não comprova equivalência. Os fatores são comparados isoladamente; os resultados não são efeitos causais ajustados por idade, popularidade ou tamanho.

CFR (b) continua sujeita à validação humana completa da heurística (#162/#163). CFR (a) mede falhas de CI, e o tempo de recuperação só existe quando há recuperação observável. As exclusões por métrica e os indicadores globais de incompletude têm os mesmos limites da #171. A janela, os valores do CSV e a classificação da heurística não são alterados por esta análise.

## Exportação e proveniência

São exportadas três tabelas novas (globais, pares completos e famílias) e um JSON com hash da fonte, hashes do código, parâmetros, fórmulas, critérios de decisão e bibliotecas. As cinco tabelas brutas e a proveniência da #171 são preservadas.

In [6]:
if hashlib.sha256(DATASET.read_bytes()).hexdigest() != sha_entrada:
    raise RuntimeError("CSV mudou durante a análise; reexecute desde o início.")
for chave, nome in ARQUIVOS.items():
    resultados[chave].to_csv(TABELAS / nome, index=False, float_format="%.17g", lineterminator="\n")
proveniencia = {
    "issue": 172,
    "csv": DATASET.relative_to(RAIZ).as_posix() if DATASET.is_relative_to(RAIZ) else str(DATASET),
    "csv_sha256": sha_entrada, "total_repositorios": len(dados),
    "janela": {"inicio": dados['janela_inicio'].iloc[0], "fim_exclusivo": dados['janela_fim_exclusivo'].iloc[0]},
    "fatores": list(FATORES), "metricas": list(METRICAS), "alpha_por_familia": ALPHA,
    "correcao": "statsmodels.stats.multitest.multipletests(method=holm); famílias global e pares separadas; ordem original",
    "familias": resultados['familias'].to_dict('records'),
    "pares_planejados": "todos os contrastes não ordenados antes da seleção por significância; teste direto de dois grupos somente na família global",
    "nao_estimaveis": "p=1 auxiliar interno para manter m planejado; saída e decisão continuam ausentes",
    "regra_relato_pares": "global_rejeita_holm E rejeita_holm do par",
    "epsilon2": "H/(n-1); H com empates; n efetivo após exclusões; só Kruskal-Wallis estimável",
    "cliff_delta": "2*U_1/(n_1*n_2)-1; positivo = valores maiores no grupo_1",
    "magnitude_delta": "abs(delta): <0.147 desprezivel; <0.33 pequeno; <0.474 medio; >=0.474 grande",
    "pares_relataveis": len(selecionados),
    "codigo_sha256": {nome: hashlib.sha256((RAIZ / nome).read_bytes()).hexdigest()
                      for nome in ('pipeline/comparacao_rq06.py', 'pipeline/inferencia_rq06.py')},
    "bibliotecas": {"python": platform.python_version(), "numpy": np.__version__, "pandas": pd.__version__,
                    "scipy": scipy.__version__, "statsmodels": statsmodels.__version__},
}
(TABELAS / "rq06_holm_proveniencia.json").write_text(json.dumps(proveniencia, ensure_ascii=False, indent=2) + "\n", encoding='utf-8')
print("Exportadas tabelas corrigidas e proveniência; dados e artefatos históricos preservados.")

Exportadas tabelas corrigidas e proveniência; dados e artefatos históricos preservados.
